In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# A Theorem Prover for First-Order Logic without Equality

## Auxiliary Functions

We need the parser for first order formulas, hence we import it.

Formulas are represented as nested arrays. In order to convert a string into a nested array we use the `LogicParser` that is found in the file `FOL-Parser.ts`. Our parser distinguishes variables and function symbols as follows:
- A word starting with a lower case letter is interpreted as a *variable*.
- A word starting with an upper case letter is assumed to be a *function* or *predicate symbol*.

In [ ]:
import { LogicParser } from './FOL-Parser';
import { RecursiveSet } from './Recursive-Set';
import { 
    parse as parseFormula, 
    normalize, 
    apply, 
    allVariables,
    formulaToString,
    prettify,
    Formula, 
    Clause, 
    CNF, 
    Literal,
    LogicalExpression,
    Substitution
} from './09-FOL-CNF';

The function `parseTerm` takes a string `s` representing a formula from first-order logic. It returns a nested array representing this formula.

In [ ]:
function parseTerm(s: string): Formula {
    const parser = new LogicParser(s);
    return parser.parse();
}

The resolution calculus works with clauses. The notebook `09-FOL-CNF.ipynb` implements the function $\texttt{normalize}(f)$ that turns a formula $f$ into a set of clauses.

In [ ]:
const s = '∀g:∀c:(Grandparent(g, c) ↔ ∃p: (Parent(g, p) ∧ Parent(p, c)))'
const f = parseFormula(s);
console.dir(f, { depth: null });

In [ ]:
const Clauses = normalize(f);
formulaToString(Clauses);

The module `unify` implements [unification](https://en.wikipedia.org/wiki/Unification_(computer_science)) via the agorithm of [Martelli and Montanari](https://dl.acm.org/doi/pdf/10.1145/357162.357169).

In [ ]:
import { unify } from './10-Unification'; 

The function call $\texttt{arb}(S)$ returns an arbitrary element from the set $S$.

In [ ]:
function arb<T>(S: RecursiveSet<T>): T | undefined {
    for (const x of S) {
        return x;
    }
    return undefined;
}

Given a literal $l$ the function $\texttt{complement}(l)$ computes the complement $\overline{\,l\,}$ of the literal $l$.

In [ ]:
function complement(l: Literal): Literal {
    if (Array.isArray(l) && l[0] === '¬') {
        return l[1] as Literal;
    }
    return ['¬', l] as Literal;
}

In [ ]:
console.log(formulaToString(complement(['P', 'x'])));

In [ ]:
console.log(formulaToString(complement(['¬', ['P', 'x']])));

Given a clause $C$, the function $\texttt{collectVariables}(C)$ computes the set of all variables occurring in $C$.  The function $\texttt{collectVariables}$ can also compute the variables occurring in a literal or a term.

In [ ]:
function collectVariables(C: LogicalExpression): RecursiveSet<string> {
    if (C instanceof RecursiveSet) {
        let vars = new RecursiveSet<string>();
        for (const literal of C) {
             vars = vars.union(collectVariables(literal as Literal));
        }
        return vars;
    }
    if (typeof C === 'string') {
        return new RecursiveSet(C);
    }
    if (Array.isArray(C)) {
        const [op, ...args] = C;
        if (op === '¬') {
            return collectVariables(args[0] as Literal);
        }
        let vars = new RecursiveSet<string>();
        for (const t of args) {
            vars = vars.union(collectVariables(t as Formula));
        }
        return vars;
    }
    return new RecursiveSet();
}

In [ ]:
for (const C of Clauses) {
    console.log(`collectVariables(${prettify(new RecursiveSet(C as Clause))}) = \n\t{${[...collectVariables(C as Clause)].join(', ')}}`);
}

In [ ]:
const asciiLowercase = 'abcdefghijklmnopqrstuvwxyz';

The function $\texttt{renameVariables}(f, g)$ takes two clauses `f` and `g` and renames the variables in the clauses `f` so that they are different from the variables occurring in `g`.

In [ ]:
function renameVariables(f: Clause, g: Clause): Clause {
    const OldVars = collectVariables(f);
    const gVars = collectVariables(g);
    const NewVarsArray = asciiLowercase.split('').filter(char => !gVars.has(char));
    const sigma: Substitution = {};
    let i = 0;
    for (const x of OldVars) {
        if (i < NewVarsArray.length) {
            sigma[x] = NewVarsArray[i++];
        } else {
             throw new Error("Not enough fresh variables for renaming available.");
        }
    }
    return apply(f, sigma) as Clause;
}

In [ ]:
for (const C of Clauses) {
    const renamed = renameVariables(C as Clause, C as Clause);
    console.log(`${prettify(new RecursiveSet(C as Clause))}  ->  ${prettify(new RecursiveSet(renamed))}`);
}

# A Calculus for First Order Logic

The [resolution](https://en.wikipedia.org/wiki/Resolution_(logic)) rule is an inference rule that is defined as follows: If
 * $C_1$ and $C_2$ are clauses from first order logic,</li>
 * $p(s_1,\cdots,s_n)$ and $p(t_1,\cdots,t_n)$ are atomic formulas,</li> 
 * the syntactical equation $p(s_1,\cdots,s_n) \doteq p(t_1,\cdots,t_n)$ is solvable and
     $$ \mu = \mathtt{mgu}\bigl(p(s_1,\cdots,s_n), p(t_1,\cdots,t_n)\bigr), $$
then
$$\frac{C_1 \cup\{ p(s_1,\cdots,s_n)\} \quad\quad \{\neg p(t_1,\cdots,t_n)\} \cup C_2}{
                 C_1\mu \cup C_2\mu} 
$$
is an application of the resolution rule.

Given a two clauses <tt>C1</tt> and <tt>C2</tt>, the function $\texttt{resolve}(\texttt{C1}, \texttt{C2})$ computes a set of all clauses that can be inferred from <tt>C1</tt> and <tt>C2</tt> by applying the resolution rule.

In [ ]:
function resolve(C1: Clause, C2: Clause): RecursiveSet<Clause> {
    const C2Renamed = renameVariables(C2, C1);
    const Result = new RecursiveSet<Clause>();
    for (const L1 of C1) {
        for (const L2 of C2Renamed) {
            const compL2 = complement(L2 as Literal);
            const mu = unify(L1 as Literal, compL2);
            if (mu !== null) {
                const sL1 = formulaToString(L1 as Literal);
                const sL2 = formulaToString(L2 as Literal);
                const C1_minus_L1 = new RecursiveSet(...[...C1].filter(l => formulaToString(l as Literal) !== sL1));
                const C2_minus_L2 = new RecursiveSet(...[...C2Renamed].filter(l => formulaToString(l as Literal) !== sL2));
                const resolvent = C1_minus_L1.union(C2_minus_L2);
                const appliedResolvent = apply(resolvent, mu) as Clause;
                Result.add(appliedResolvent);
            }
        }
    }
    return Result;
}

## Some Formulas for Testing

According to <a href="https://de.wikipedia.org/wiki/Uwe_Schöning">Uwe Schöning</a>, the theory of red dragons is
given by the following axioms:
<ol>
<li>
Every dragon is happy if all its children can fly:
$$ \forall x: \Bigl(\forall y: \big(\texttt{Child}(y,x) \rightarrow \texttt{CanFly}(y)\big) \rightarrow \texttt{Happy}(x)\Bigr) 
$$
</li>
<li> 
All red dragons can fly:
$$
 \forall x: \bigl(\texttt{Red}(x) \rightarrow \texttt{CanFly}(x)\bigr)
$$
</li>
<li> The children of red dragons are themselves red:
$$
\forall x: \bigl(\texttt{Red}(x) \rightarrow \forall y:\bigl( \texttt{Child}(y,x) \rightarrow \texttt{Red}(y)\bigr)\bigr)
$$
</li>
</ol>
We will show that these axioms imply that all red dragons are happy:
$$
 \forall x: \bigl(\texttt{Red}(x) \rightarrow \texttt{Happy}(x)\bigr)
$$
To this end, the formula stating that all red dragons can fly is negated.  Then we will show that the set consisting of the negated formula together with the axioms is inconsistent.  We start by defining the formulas.

In [ ]:
const s1 = '∀x:(∀y:(Child(y, x) → CanFly(y)) → Happy(x))';
const s2 = '∀x:(Red(x) → CanFly(x))';
const s3 = '∀x:(Red(x) → ∀y:(Child(y, x) → Red(y)))';
const s4 = '¬∀x:(Red(x) → Happy(x))';

Next, the formulas are parsed and transformed into clauses.

In [ ]:
const f1 = parseFormula(s1);
formulaToString(normalize(f1));

In [ ]:
const f2 = parseFormula(s2);
formulaToString(normalize(f2));

In [ ]:
const f3 = parseFormula(s3);
formulaToString(normalize(f3));

In [ ]:
const f4 = parseFormula(s4);
formulaToString(normalize(f4));

In [ ]:
let Clauses = normalize(f1)
    .union(normalize(f2))
    .union(normalize(f3))
    .union(normalize(f4));
    
console.log(prettify(Clauses));

We give names to the clauses in order to be able to refer to them.

In [ ]:
const C1: Clause = new RecursiveSet(['Red', ['sk3']]);

In [ ]:
const C2: Clause = new RecursiveSet(['¬', ['Happy', ['sk3']]]);

In [ ]:
const C3: Clause = new RecursiveSet(['CanFly', 'x'], ['¬', ['Red', 'x']]);

In [ ]:
const C4: Clause = new RecursiveSet(['Child', ['sk2', 'x'], 'x'], ['Happy', 'x']);

In [ ]:
const C5: Clause = new RecursiveSet(['Happy', 'x'], ['¬', ['CanFly', ['sk2', 'x']]]);

In [ ]:
const C6: Clause = new RecursiveSet(['Red', 'y'], ['¬', ['Child', 'y', 'x']], ['¬', ['Red', 'x']]);

Now we are ready to show that the set consisting of these clauses is inconsistent.

In [ ]:
const C7 = arb(resolve(C1, C6));
formulaToString(C7); 

In [ ]:
const C8 = arb(resolve(C7, C4))
formulaToString(C8)

In [ ]:
const C9 = arb(resolve(C8, C2))
formulaToString(C9)

In [ ]:
const C10 = arb(resolve(C9, C3))
formulaToString(C10)

In [ ]:
const C11 = arb(resolve(C10, C5))
formulaToString(C11)

In [ ]:
arb(resolve(C11, C2))

As we have derived the empty set, we have shown that all <b style="color:red;">communist dragons</b> are happy!

## Factorization

A calculus which only contains the resolution rule is not complete. We also need the factorization rule. If
- $C$ is a clause from first order logic,
- $p(s_1,\cdots,s_n)$ and $p(t_1,\cdots,t_n)$ are atomic formulas,
- the syntactical equation $p(s_1,\cdots,s_n) \doteq p(t_1,\cdots,t_n)$ is solvable and 
$$\mu = \mathtt{mgu}\bigl(p(s_1,\cdots,s_n), p(t_1,\cdots,t_n)\bigr),$$

then both 

$$
\displaystyle \frac{C \cup \bigl\{p(s_1,\cdots,s_n),\, p(t_1,\cdots,t_n)\bigr\}}{C\mu \cup \bigl\{p(s_1,\cdots,s_n)\mu\bigr\}} 
$$ 

and 

$$\displaystyle \frac{C \cup \bigl\{ \neg p(s_1,\cdots,s_n),\, \neg p(t_1,\cdots,t_n)\bigr\}}{C\mu \cup \bigl\{\neg p(s_1,\cdots,s_n)\mu\bigr\}}$$

are applications of the factorization rule.

The function $\texttt{factorize}(C)$ takes a clause $C$ from first order logic and computes all clauses that can be derived from $C$ via factorization.

In [ ]:
function factorize(C: Clause): RecursiveSet<Clause> {
    const Result = new RecursiveSet<Clause>();
    const literals = [...C] as Literal[];
    
    for (let i = 0; i < literals.length; i++) {
        for (let j = i + 1; j < literals.length; j++) {
            const L1 = literals[i];
            const L2 = literals[j];
            
            const mu = unify(L1, L2);
            
            if (mu !== null) {
                const Cmu = apply(C, mu) as Clause;
                Result.add(Cmu);
            }
        }
    }
    return Result;
}

The clauses 
$$C_1 := \forall x: \forall y: P(F(x),y) \vee \forall u: \forall v:P(u, G(v))$$
and
$$C_2 := \forall x: \forall y: \bigl(\neg P(F(x),y)\bigr) \vee \forall u: \forall v: \bigl(\neg P(u, G(v))\bigr)$$
are inconsistent. However, the resolution rule alone is not sufficient to show this.

In [ ]:
const C1 = arb(normalize(parseFormula('∀x:∀y:P(F(x),y) ∨ ∀u:∀v:P(u,G(v))')))
formulaToString(C1)

In [ ]:
const C2 = arb(normalize(parseFormula('∀x:∀y:(¬P(F(x),y)) ∨ ∀u:∀v:(¬P(u,G(v)))')))
formulaToString(C2)

In [ ]:
const C3 = arb(factorize(C1))
formulaToString(C3)

In [ ]:
const C4 = arb(factorize(C2))
formulaToString(C4)

In [ ]:
arb(resolve(C3, C4))

## Automatic Theorem Proving

The function $\texttt{infere}(\texttt{Clauses})$ returns all possible clauses that result from:
- the resolution of two clauses $C_1, C_2 \in \texttt{Clauses}$,
- the factorization of a clause $C \in \texttt{Clauses}$.

In [ ]:
type ReasonMap = Map<string, Clause[]>;

function infere(Clauses: CNF): { newClauses: CNF, reasons: ReasonMap } {
    const newClauses = new RecursiveSet<Clause>();
    const reasons: ReasonMap = new Map();
    const clausesArray = [...Clauses] as Clause[];
    for (let i = 0; i < clausesArray.length; i++) {
        for (let j = 0; j < clausesArray.length; j++) {
             if (i === j) continue; 
             const C1 = clausesArray[i];
             const C2 = clausesArray[j];
             const resolvents = resolve(C1, C2);
             for (const res of resolvents) {
                 newClauses.add(res);
                 const key = res.toString();
                 if (!reasons.has(key)) {
                     reasons.set(key, [C1, C2]);
                 }
             }
        }
    }
    for (const C of Clauses) {
        const factors = factorize(C as Clause);
        for (const factor of factors) {
             newClauses.add(factor);
             
             const key = factor.toString();
             if (!reasons.has(key)) {
                 reasons.set(key, [C as Clause]);
             }
        }
    }
    return { newClauses, reasons };
}

In [ ]:
function prettyPrint(Clauses: CNF): void {
    console.log(prettify(Clauses));
}

In [ ]:
prettyPrint(Clauses);

The function $\texttt{saturateWithProof}(\texttt{Cs})$ takes a set of clauses $\texttt{Cs}$ as input and tries to infer the empty clause. If it is not possible to infer the empty clause, the function runs until saturation is reached or memory is exhausted.

In [ ]:
function saturateWithProof(Cs: CNF): ReasonMap {
    let Clauses = new RecursiveSet(...Cs);
    let cnt = 1;
    const Reasons: ReasonMap = new Map(); 
    while (true) {
        for (const C of Clauses) {
            if (C.size === 0) {
                console.log("Empty clause found!");
                return Reasons; 
            }
        }
        const result = infere(Clauses); 
        const newClauses = result.newClauses;
        const stepReasons = result.reasons;
        let newAddedCount = 0;
        for (const C of newClauses) {
            if (!Clauses.has(C)) {
                Clauses.add(C);
                newAddedCount++;
                const cKey = C.toString();
                const parents = stepReasons.get(cKey);
                if (parents && !Reasons.has(cKey)) {
                    Reasons.set(cKey, parents);
                }
            }
        }
        console.log(`cnt = ${cnt}, number of clauses: ${Clauses.size}, new: ${newAddedCount}`);
        cnt++;
        if (newAddedCount === 0) {
            console.log("Saturation reached (no empty clause found).");
            return Reasons;
        }
    }
}

In [ ]:
console.time("saturate")
const proofReasons = saturateWithProof(Clauses);
console.timeEnd("saturate")

In [ ]:
function updateProof(p1: string[], p2: string[]): string[] {
    const res = [...p1];
    for (const line of p2) {
        if (!res.includes(line)) {
            res.push(line);
        }
    }
    return res;
}

Given a dictionary $\texttt{Reasons}$ and a clause $\texttt{clause}$, the function `constructProof` returns a proof of $\texttt{clause}$.

In [ ]:
function constructProof(clause: Clause, Reasons: ReasonMap): string[] {
    const clauseKey = clause.toString();
    const clauseStr = formulaToString(clause); 
    if (Reasons.has(clauseKey)) {
        const parents = Reasons.get(clauseKey)!; 
        if (parents.length === 1) {
            const parent = parents[0];
            const parentStr = formulaToString(parent);
            const parentProof = constructProof(parent, Reasons);
            return updateProof(parentProof, [
                `Factorization: ${parentStr}`,
                `             ⊢ ${clauseStr}`
            ]);
        } else if (parents.length === 2) {
            const [p1, p2] = parents;
            const p1Str = formulaToString(p1);
            const p2Str = formulaToString(p2);
            const proof1 = constructProof(p1, Reasons);
            const proof2 = constructProof(p2, Reasons);
            let combined = updateProof(proof1, proof2);
            combined.push(`Resolution: ${p1Str},`);
            combined.push(`            ${p2Str}`);
            combined.push(`          ⊢ ${clauseStr}`);
            return combined;
        }
    }
    return [`Axiom: ${clauseStr}`];
}

In [ ]:
const emptyClause = new RecursiveSet<Literal>();
const proofLines = constructProof(emptyClause, proofReasons);
for (const line of proofLines) {
    console.log(line);
}

In [ ]:
prettify(Clauses);